# Predicting Water Well Functionality in Tanzania
### A Classification Model for the Tanzanian Ministry of Water

**Author:** [Your Name]  
**Date:** [Date]  
**Dataset:** DrivenData — Pump It Up: Data Mining the Water Table

---
## 1. Business Understanding

### 1.1 Problem Statement

Tanzania is an East African country of over 60 million people, many of whom depend on water wells (waterpoints) for daily survival. A significant portion of these wells are non-functional or in need of repair — leaving communities without access to clean, potable water.

Manually inspecting every well across a vast country is expensive and slow. A predictive model that identifies which wells are likely to be non-functional or in need of repair would allow authorities to **prioritize maintenance resources efficiently**, saving costs and improving water access at scale.

### 1.2 Stakeholders

- **Primary Stakeholder:** The Tanzanian Ministry of Water — responsible for water infrastructure and maintenance operations nationwide.
- **Secondary Stakeholders:** NGOs (e.g., UNICEF, Danida) that fund and manage well installations and repairs.

### 1.3 How the Model Will Be Used

Given a set of well attributes (location, pump type, funder, construction year, etc.), the model will predict one of three classes:
- `functional` — well is working, no action needed
- `functional needs repair` — well is working but at risk; schedule maintenance
- `non functional` — well is broken; prioritize for urgent repair or replacement

This enables the Ministry to shift from **reactive** to **proactive** maintenance, reducing downtime and the number of communities left without water.

### 1.4 Success Metric

We will optimize for **recall on non-functional wells** — because the cost of missing a broken well (leaving a community without water) is far greater than the cost of a false alarm (sending a technician to inspect a functioning well).

---
## 2. Imports & Setup

In [ ]:
# Core libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Scikit-learn — preprocessing
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

# Scikit-learn — models
from sklearn.dummy import DummyClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression

# Scikit-learn — evaluation
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    recall_score,
    accuracy_score
)

# Display settings
pd.set_option('display.max_columns', 50)
pd.set_option('display.max_rows', 100)
sns.set_style('whitegrid')
%matplotlib inline

import warnings
warnings.filterwarnings('ignore')

---
## 3. Data Understanding

### 3.1 Data Source

The data comes from **Taarifa** and the **Tanzanian Ministry of Water**, hosted on DrivenData as part of the [Pump It Up competition](https://www.drivendata.org/competitions/7/pump-it-up-data-mining-the-water-table/).

Three files are provided:
- `training_set_values.csv` — 59,400 rows × 40 feature columns
- `training_set_labels.csv` — 59,400 rows × target column (`status_group`)
- `test_set_values.csv` — 14,850 rows (no labels; for competition submission only)

We will use the training set for all modeling, splitting it ourselves into train/validation/test sets.

In [ ]:
# Load data
# Update paths to wherever you saved the CSVs
X_raw = pd.read_csv('data/training_set_values.csv', index_col='id')
y_raw = pd.read_csv('data/training_set_labels.csv', index_col='id')

# Merge into one DataFrame for EDA
df = X_raw.join(y_raw)

print(f'Dataset shape: {df.shape}')
df.head()

### 3.2 Target Variable Distribution

In [ ]:
# Class balance
class_counts = df['status_group'].value_counts()
class_pct = df['status_group'].value_counts(normalize=True) * 100

print('Class Counts:')
print(class_counts)
print('\nClass Percentages:')
print(class_pct.round(1))

# Plot
fig, ax = plt.subplots(figsize=(8, 4))
class_counts.plot(kind='bar', ax=ax, color=['#2ecc71', '#f39c12', '#e74c3c'])
ax.set_title('Distribution of Well Status (Target Variable)', fontsize=14)
ax.set_xlabel('Status Group')
ax.set_ylabel('Count')
ax.tick_params(axis='x', rotation=20)
plt.tight_layout()
plt.show()

### 3.3 Feature Overview & Descriptive Statistics

In [ ]:
# Data types overview
print('Data Types:')
print(df.dtypes.value_counts())
print('\nNumerical Summary:')
df.describe()

In [ ]:
# Missing values
missing = df.isnull().sum()
missing_pct = (missing / len(df)) * 100
missing_df = pd.DataFrame({'missing_count': missing, 'missing_pct': missing_pct})
missing_df = missing_df[missing_df['missing_count'] > 0].sort_values('missing_pct', ascending=False)

print('Columns with Missing Values:')
print(missing_df)

### 3.4 Exploratory Data Analysis (EDA)

> **Note:** EDA should guide feature selection and preprocessing decisions. Explore distributions, relationships with the target, and anomalies.

In [ ]:
# --- Example: Construction Year ---
# Many wells have construction_year = 0 (invalid — likely missing)
print('Construction year = 0:', (df['construction_year'] == 0).sum())

valid_years = df[df['construction_year'] > 0]
fig, ax = plt.subplots(figsize=(10, 4))
for status, color in zip(['functional', 'functional needs repair', 'non functional'],
                          ['#2ecc71', '#f39c12', '#e74c3c']):
    subset = valid_years[valid_years['status_group'] == status]
    ax.hist(subset['construction_year'], bins=30, alpha=0.6, label=status, color=color)
ax.set_title('Construction Year by Well Status')
ax.set_xlabel('Construction Year')
ax.set_ylabel('Count')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# --- Example: Water Quantity vs Status ---
pd.crosstab(df['quantity'], df['status_group'], normalize='index').plot(
    kind='bar', stacked=True, figsize=(10, 5),
    color=['#2ecc71', '#f39c12', '#e74c3c']
)
plt.title('Well Status by Water Quantity')
plt.ylabel('Proportion')
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

In [ ]:
# --- Example: Geographic distribution ---
fig, ax = plt.subplots(figsize=(10, 7))
colors = {'functional': '#2ecc71', 'functional needs repair': '#f39c12', 'non functional': '#e74c3c'}
for status, color in colors.items():
    subset = df[df['status_group'] == status]
    ax.scatter(subset['longitude'], subset['latitude'], s=0.3, alpha=0.3,
               color=color, label=status)
ax.set_title('Geographic Distribution of Well Status')
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
ax.legend(markerscale=10)
plt.tight_layout()
plt.show()

---
## 4. Data Preparation

### 4.1 Feature Selection & Rationale

The raw dataset has 40 features. We will remove:
- **Duplicate/redundant columns** (e.g., `region_code` and `region` both encode geography)
- **High-cardinality ID-like columns** with no predictive value (e.g., `wpt_name`, `subvillage`)
- **Data leakage risks** (columns that would not be known at prediction time)

Features we'll retain and justify are documented below.

In [ ]:
# Columns to drop — justify each removal
cols_to_drop = [
    'wpt_name',        # free-text name, near-unique, no predictive value
    'num_private',     # mostly zeros, unclear meaning
    'subvillage',      # extremely high cardinality (~19,000 unique values)
    'region_code',     # redundant with 'region'
    'district_code',   # redundant with 'lga'
    'recorded_by',     # single unique value — no variance
    'scheme_name',     # very high cardinality and many nulls
    'extraction_type', # redundant with 'extraction_type_group'
    'payment',         # redundant with 'payment_type'
    'water_quality',   # redundant with 'quality_group'
    'quantity_group',  # redundant with 'quantity'
    'source_type',     # redundant with 'source'
    'waterpoint_type_group',  # redundant with 'waterpoint_type'
    'management',      # redundant with 'management_group'
]

X = X_raw.drop(columns=cols_to_drop)
y = y_raw['status_group']

print(f'Features remaining: {X.shape[1]}')
print(X.columns.tolist())

### 4.2 Train / Validation / Test Split

We split **before** any fitting (imputers, encoders, scalers) to prevent data leakage.

In [ ]:
# First split: 80% train+val, 20% holdout test
X_trainval, X_test, y_trainval, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Second split: 75% train, 25% val (of the 80%)
X_train, X_val, y_train, y_val = train_test_split(
    X_trainval, y_trainval, test_size=0.25, random_state=42, stratify=y_trainval
)

print(f'Train:      {X_train.shape[0]:,} rows')
print(f'Validation: {X_val.shape[0]:,} rows')
print(f'Test:       {X_test.shape[0]:,} rows (held out — do not touch until final evaluation!)')

### 4.3 Preprocessing

**Missing values:**
- Numerical: impute with median (robust to outliers)
- Categorical: impute with constant `'unknown'`
- `construction_year == 0` treated as missing

**Encoding:**
- Categorical features → Label Encoding (suitable for tree-based models)

**Scaling:**
- Applied only for logistic regression; tree models don't require it

In [ ]:
def preprocess(X_train, X_val_or_test, construction_year_col='construction_year'):
    """
    Fit preprocessing on X_train, transform both sets.
    Returns transformed copies.
    """
    X_train = X_train.copy()
    X_val_or_test = X_val_or_test.copy()

    # Treat construction_year == 0 as missing
    X_train[construction_year_col] = X_train[construction_year_col].replace(0, np.nan)
    X_val_or_test[construction_year_col] = X_val_or_test[construction_year_col].replace(0, np.nan)

    # Separate numeric and categorical columns
    num_cols = X_train.select_dtypes(include='number').columns.tolist()
    cat_cols = X_train.select_dtypes(include='object').columns.tolist()

    # Impute numeric (fit on train only)
    num_imputer = SimpleImputer(strategy='median')
    X_train[num_cols] = num_imputer.fit_transform(X_train[num_cols])
    X_val_or_test[num_cols] = num_imputer.transform(X_val_or_test[num_cols])

    # Impute categorical
    cat_imputer = SimpleImputer(strategy='constant', fill_value='unknown')
    X_train[cat_cols] = cat_imputer.fit_transform(X_train[cat_cols])
    X_val_or_test[cat_cols] = cat_imputer.transform(X_val_or_test[cat_cols])

    # Label encode categoricals (fit on train)
    encoders = {}
    for col in cat_cols:
        le = LabelEncoder()
        X_train[col] = le.fit_transform(X_train[col].astype(str))
        # Handle unseen labels in val/test
        X_val_or_test[col] = X_val_or_test[col].astype(str).apply(
            lambda x: le.transform([x])[0] if x in le.classes_ else -1
        )
        encoders[col] = le

    return X_train, X_val_or_test, encoders


X_train_proc, X_val_proc, encoders = preprocess(X_train, X_val)
print('Preprocessing complete.')
print(f'Train shape: {X_train_proc.shape}, Val shape: {X_val_proc.shape}')

---
## 5. Modeling

We iterate from a simple baseline to increasingly complex models. Each iteration is justified by the results of the prior model.

### 5.1 Baseline Model — Dummy Classifier

A baseline using the most frequent class gives us a performance floor. Any real model must beat this.

In [ ]:
dummy = DummyClassifier(strategy='most_frequent', random_state=42)
dummy.fit(X_train_proc, y_train)

y_pred_dummy = dummy.predict(X_val_proc)
print('=== Baseline: Dummy Classifier (Most Frequent) ===')
print(f'Accuracy: {accuracy_score(y_val, y_pred_dummy):.3f}')
print(classification_report(y_val, y_pred_dummy))

### 5.2 Model 2 — Decision Tree

**Rationale:** A decision tree is interpretable and a natural first step for multi-class classification. It also reveals which features are most predictive, informing our next iteration.

In [ ]:
dt = DecisionTreeClassifier(max_depth=10, random_state=42)
dt.fit(X_train_proc, y_train)

y_pred_dt = dt.predict(X_val_proc)
print('=== Model 2: Decision Tree (max_depth=10) ===')
print(f'Accuracy: {accuracy_score(y_val, y_pred_dt):.3f}')
print(classification_report(y_val, y_pred_dt))

In [ ]:
# Feature importance from Decision Tree
feat_imp = pd.Series(dt.feature_importances_, index=X_train_proc.columns)
feat_imp.nlargest(15).plot(kind='barh', figsize=(8, 6))
plt.title('Top 15 Feature Importances — Decision Tree')
plt.xlabel('Importance')
plt.tight_layout()
plt.show()

### 5.3 Model 3 — Random Forest

**Rationale:** The Decision Tree likely overfits. A Random Forest uses ensemble averaging across many trees to reduce variance and improve generalization, especially on noisy, high-cardinality categorical data like this dataset.

In [ ]:
rf = RandomForestClassifier(n_estimators=100, max_depth=20, random_state=42, n_jobs=-1)
rf.fit(X_train_proc, y_train)

y_pred_rf = rf.predict(X_val_proc)
print('=== Model 3: Random Forest ===')
print(f'Accuracy: {accuracy_score(y_val, y_pred_rf):.3f}')
print(classification_report(y_val, y_pred_rf))

### 5.4 Model 4 — Tuned Random Forest

**Rationale:** The baseline Random Forest may not be using optimal hyperparameters. We use GridSearchCV to tune key parameters identified from the prior model's performance gaps.

In [ ]:
# NOTE: GridSearchCV can be slow — reduce param_grid if needed
param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [15, 20, None],
    'min_samples_leaf': [1, 2, 4],
}

rf_tuned = GridSearchCV(
    RandomForestClassifier(random_state=42, n_jobs=-1),
    param_grid=param_grid,
    cv=3,
    scoring='recall_macro',  # Recall across all 3 classes equally weighted
    verbose=1
)
rf_tuned.fit(X_train_proc, y_train)

print('Best params:', rf_tuned.best_params_)
y_pred_rf_tuned = rf_tuned.predict(X_val_proc)
print('=== Model 4: Tuned Random Forest ===')
print(f'Accuracy: {accuracy_score(y_val, y_pred_rf_tuned):.3f}')
print(classification_report(y_val, y_pred_rf_tuned))

### 5.5 Model Comparison Summary

In [ ]:
models = {
    'Dummy Classifier': y_pred_dummy,
    'Decision Tree': y_pred_dt,
    'Random Forest': y_pred_rf,
    'Tuned Random Forest': y_pred_rf_tuned,
}

results = []
for name, preds in models.items():
    results.append({
        'Model': name,
        'Accuracy': accuracy_score(y_val, preds),
        'Macro Recall': recall_score(y_val, preds, average='macro'),
    })

results_df = pd.DataFrame(results).set_index('Model')
print(results_df.round(3))

---
## 6. Evaluation

### 6.1 Final Model Selection

Based on validation performance, we select the **Tuned Random Forest** as our final model because it achieves the highest macro recall — meaning it best identifies wells in all three states, including the critical non-functional class.

### 6.2 Final Evaluation on Holdout Test Set

> ⚠️ **This cell should only be run ONCE — at the very end of the project.**

In [ ]:
# Preprocess test set using the same encoders fitted on train
_, X_test_proc, _ = preprocess(X_train, X_test)

# Final prediction
y_pred_final = rf_tuned.predict(X_test_proc)

print('=== FINAL MODEL EVALUATION ON HOLDOUT TEST SET ===')
print(f'Accuracy:     {accuracy_score(y_test, y_pred_final):.3f}')
print(f'Macro Recall: {recall_score(y_test, y_pred_final, average="macro"):.3f}')
print()
print(classification_report(y_test, y_pred_final))

In [ ]:
# Confusion Matrix
cm = confusion_matrix(y_test, y_pred_final, labels=rf_tuned.classes_)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=rf_tuned.classes_)
fig, ax = plt.subplots(figsize=(8, 6))
disp.plot(ax=ax, cmap='Blues', colorbar=False)
ax.set_title('Confusion Matrix — Final Model (Test Set)', fontsize=13)
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

### 6.3 Interpretation & Business Implications

> **Fill this in after running evaluation.**

Key questions to address:
- How well does the model identify non-functional wells? What is the recall for that class?
- What is the practical cost of false negatives (missed broken wells)?
- Which features matter most? Do they make real-world sense?
- Where does the model struggle (e.g., `functional needs repair`)? Why might that be?
- What actions should the Ministry of Water take based on these predictions?

---
## 7. Conclusion

### Summary

In this project, we built a multi-class classifier to predict the operational status of water wells across Tanzania. Using data from the Tanzanian Ministry of Water, we trained a Tuned Random Forest that achieves strong performance on holdout test data.

### Recommendations for the Ministry of Water

1. **Deploy the model for triage:** Input well attributes into the model to flag non-functional and at-risk wells, enabling data-driven maintenance scheduling.
2. **Prioritize high-recall regions:** Focus inspection resources on regions where the model predicts high concentrations of non-functional wells.
3. **Track construction year:** Older wells with unknown construction years should be treated as higher risk and inspected sooner.

### Limitations

- **Class imbalance:** `functional needs repair` wells are underrepresented and harder to predict — more labeled data for this class would improve performance.
- **Data staleness:** Well status changes over time; the model should be retrained periodically with updated survey data.
- **Missing funder/installer data:** High proportions of missing values in these fields limit their predictive power.

### Next Steps

- Try XGBoost or LightGBM, which often perform better on high-cardinality categorical data
- Explore SMOTE or class weighting to improve recall on the minority class
- Develop a cost-benefit function to prioritize interventions based on repair cost vs. impact